# Qwen3-TTS Voice Clone

Clone any voice with a short reference audio using [Qwen3-TTS](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base).

**How to use:**
1. Click **Runtime → Run all** (or press `Ctrl+Shift+F9`)
2. Choose your model and enter the text you want the cloned voice to say
3. Upload a reference audio file when prompted
4. Wait for the voice clone to generate — it will play and download automatically

**Requirements:** T4 GPU runtime (free tier)

## Block 1 — GPU check

In [ ]:
import torch

# Fail fast if no GPU — the models require CUDA for reasonable inference speed
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then re-run."
    )

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Block 2 — Install dependencies

In [ ]:
# sox is a system-level audio library required by qwen-tts for audio processing
!apt-get install -y -q sox

# Install qwen-tts and qwen-asr following the official Qwen documentation
!pip install -U qwen-tts
!pip install -U qwen-asr

print("✅ Dependencies installed")

## Block 3 — Import modules

In [ ]:
import torch
import soundfile as sf          # Read/write audio files (WAV, FLAC, etc.)
import librosa                  # Audio loading and resampling
import numpy as np              # Numerical operations (used for audio validation)
from qwen_tts import Qwen3TTSModel       # Text-to-speech model
from qwen_asr import Qwen3ASRModel       # Speech recognition model (for auto-transcription)
from IPython.display import Audio, display  # Play audio in Colab
from google.colab import files           # File upload/download in Colab
import os
import time

# Determine device — all model loading uses this variable
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Imports loaded")

## Block 4 — Configure

Edit these settings using the form widgets below, or change the values directly.

- **TTS Model:** 1.7B = higher quality, 0.6B = faster and uses less VRAM
- **Text to speak:** The text the cloned voice will say

In [ ]:
#@title Voice cloning settings

#@markdown **TTS Model** — 1.7B is higher quality, 0.6B is faster and uses less VRAM
model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-Base" #@param ["Qwen/Qwen3-TTS-12Hz-0.6B-Base", "Qwen/Qwen3-TTS-12Hz-1.7B-Base"]

#@markdown **Text to speak** — what should the cloned voice say?
generated_text = "Hello! This is a test of the Qwen3 voice cloning system. The voice you hear was generated from a short reference audio sample." #@param {type:"string"}

#@markdown ---
print(f"Model: {model_name}")
print(f"Text: {generated_text[:80]}...")

## Block 5 — Upload reference audio

Upload a voice recording (.wav, .mp3, etc.) of the person whose voice you want to clone.

**Tips for best results:**
- 3–10 seconds of clear, single-speaker speech
- No background music, noise, or reverb
- Speaker should be speaking at a normal pace

In [ ]:
# File paths used throughout the notebook
REFERENCE_VOICE_FILE = "reference_voice.wav"   # Input: the voice to clone
OUTPUT_VOICE_FILE = "cloned_voice.wav"         # Output: the generated speech

# Upload the reference audio file via Colab's file picker
print("📁 Upload a reference audio file...")
uploaded_files = files.upload()

if not uploaded_files:
    raise SystemExit("No file uploaded — re-run this cell to try again.")

# Rename to a consistent filename (avoids issues with spaces/special chars in original name)
original_filename = list(uploaded_files.keys())[0]
os.rename(original_filename, REFERENCE_VOICE_FILE)

# Validate the audio to catch common issues before running the model
# Bad input = bad output, so we check early and warn clearly
print("\n🔍 Validating...")
audio_check, sr_check = librosa.load(REFERENCE_VOICE_FILE, sr=None, mono=True)
duration = len(audio_check) / sr_check
rms = np.sqrt(np.mean(audio_check**2))  # Root mean square = perceived loudness

if duration < 2:
    print(f"⚠️  Only {duration:.1f}s — recommend 3–10s for best results")
elif duration > 30:
    print(f"⚠️  {duration:.1f}s — recommend 3–10s, longer clips may have noise")
else:
    print(f"✅ Duration: {duration:.1f}s")

if rms < 0.001:
    raise ValueError("Audio appears silent (RMS < 0.001). Upload a file with audible speech.")
else:
    print(f"✅ Audio level OK")

# Play back so the user can verify the upload is correct
print("\n📻 Reference audio:")
display(Audio(REFERENCE_VOICE_FILE))

## Block 6 — Auto-transcribe reference audio

The TTS model needs to know *what words are spoken* in the reference audio to extract the speaker's voice characteristics accurately. Instead of asking you to type this manually, we use Qwen3-ASR to transcribe it automatically.

After transcription, the ASR model is unloaded to free GPU memory for the TTS model.

In [ ]:
# Use the smaller 0.6B ASR model — fast and accurate enough for transcription
# bfloat16 uses half the memory of float32 with minimal quality loss on GPU
asr_model_name = "Qwen/Qwen3-ASR-0.6B"
asr_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading ASR model: {asr_model_name}")
asr_model = Qwen3ASRModel.from_pretrained(asr_model_name, dtype=asr_dtype, device_map="auto")

# Transcribe the reference audio
# language=None lets the model auto-detect the language
# inference_mode() disables gradient tracking to save memory and speed up
print("🎤 Transcribing reference audio...")
try:
    with torch.inference_mode():
        results = asr_model.transcribe(audio=REFERENCE_VOICE_FILE, language=None, return_time_stamps=False)
finally:
    # Always free GPU memory, even if transcription fails
    del asr_model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 ASR model unloaded")

# Guard against empty transcription (silent audio, unrecognisable speech)
if not results or not results[0].text.strip():
    raise ValueError("ASR returned empty transcription — upload a clearer audio file")

# Let the user confirm or edit the transcription
# Wrong ref_text silently degrades voice quality — this is the #1 quality killer
reference_text = results[0].text.strip()
print(f"
📝 Transcribed: "{reference_text}"")
print("   If this looks wrong, edit it below for better results.")
user_input = input("   Confirm/edit (press Enter to keep): ").strip()
if user_input:
    reference_text = user_input
print(f"✅ Using: {reference_text}")

## Block 7 — Load TTS model

In [ ]:
# Warn if VRAM is tight — the 1.7B model needs ~4GB but can spike higher with long text
if device == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if "1.7B" in model_name and vram_gb < 16:
        print(f"⚠️  VRAM: {vram_gb:.1f}GB — 1.7B model may OOM. Consider switching to 0.6B.")

# Use bfloat16 on GPU (saves memory, same quality), float16 fallback, float32 on CPU
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading TTS model: {model_name}")
model = Qwen3TTSModel.from_pretrained(
    model_name,
    dtype=dtype,
    device_map="auto"
)
print("✅ TTS model loaded")

## Block 8 — Generate cloned voice

In [ ]:
def detect_language(text):
    # Detect if text is primarily Chinese or English
    # Heuristic: if >30% of characters are CJK ideographs, it's Chinese
    if not text.strip():
        return "English"  # Default to English for empty input
    chinese_chars = sum(1 for c in text if "\u4e00" <= c <= "\u9fff")
    return "Chinese" if chinese_chars / len(text) > 0.3 else "English"

# Guard against empty text — undefined behaviour on model.generate_voice_clone(text="")
if not generated_text.strip():
    raise ValueError("Text to speak is empty — enter some text in Block 4.")

target_language = detect_language(generated_text)

print(f"🎵 Generating voice clone...")
print(f"   Language: {target_language}")
print(f"   Reference: {reference_text}")

start = time.time()

try:
    # inference_mode() disables gradient tracking — saves memory and speeds up generation
    with torch.inference_mode():
        wavs, sr = model.generate_voice_clone(
            text=generated_text,           # What to say
            language=target_language,       # Language for phoneme generation
            ref_audio=REFERENCE_VOICE_FILE, # The voice to clone
            ref_text=reference_text         # Transcript of the reference (from ASR)
        )
except Exception as e:
    print(f"❌ Voice cloning failed: {e}")
    print("   Try a shorter audio file or switch to the 0.6B model")
    raise

elapsed = time.time() - start
print(f"✅ Done in {elapsed:.1f}s — {len(wavs[0]) / sr:.2f}s of audio")

## Block 9 — Save and download

In [ ]:
# Save the generated audio to a WAV file
# wavs[0] = first (and only) waveform from the batch
# sr = sample rate (24kHz for Qwen3-TTS)
sf.write(OUTPUT_VOICE_FILE, wavs[0], sr)
print(f"💾 Saved as {OUTPUT_VOICE_FILE}")

# Play it back in Colab so you can listen immediately
display(Audio(OUTPUT_VOICE_FILE))

# Trigger browser download — if it fails, user can grab it from the Files panel
try:
    files.download(OUTPUT_VOICE_FILE)
    print("✅ Downloading...")
except Exception:
    print(f"Manual download: Files panel → {OUTPUT_VOICE_FILE}")